# Sprint 3 — Decision Tree (baseline)

A new notebook is a new kernel: it cannot see `02`'s variables. We load the **persisted split** from `data/processed/` (`foundation.md Section 10`) instead of re-cleaning 1.7 GB from raw — the same rows `02` produced, so Sprint 5's SVM will compare on the identical data (`Section 7 #9`).

This is the **baseline** of `Section 7 #12`: a plain tree on the real 80/20 — **no `class_weight`, no resampling**. The point is to *measure* the imbalance trap before treating it.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

proc = Path('../data/processed')
X_train = pd.read_parquet(proc / 'X_train.parquet')
X_test  = pd.read_parquet(proc / 'X_test.parquet')
y_train = pd.read_parquet(proc / 'y_train.parquet')['label_binary']
y_test  = pd.read_parquet(proc / 'y_test.parquet')['label_binary']

# self-check: must match exactly what 02 wrote (foundation.md Section 7 #9)
print('train:', X_train.shape, '| test:', X_test.shape)
print('malicious frac  ->  train: %.4f   test: %.4f' % (y_train.mean(), y_test.mean()))
print('NaN in features:', int(X_train.isna().to_numpy().sum() + X_test.isna().to_numpy().sum()))

train: (2264502, 65) | test: (566126, 65)
malicious frac  ->  train: 0.1970   test: 0.1970
NaN in features: 0


## 1. The "always benign" model — the Section 11 trap, made concrete

Before training anything, build the model that does **nothing**: predict the majority class for every flow. It is literally `return 0`.

Watch its two numbers. Accuracy will look respectable. Recall on malicious will be **zero** — it catches no intrusions at all. That pairing is the failure this project exists to refuse (`foundation.md Section 11`).

In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

dummy = DummyClassifier(strategy='most_frequent')   # always predict the majority class (benign)
dummy.fit(X_train, y_train)
y_dummy = dummy.predict(X_test)

n_attacks = int((y_test == 1).sum())
print('accuracy           : %.4f   <- looks fine' % accuracy_score(y_test, y_dummy))
print('recall (malicious) : %.4f   <- catches nothing' % recall_score(y_test, y_dummy, zero_division=0))
print('attacks caught     : %d of %d' % (int(((y_dummy == 1) & (y_test == 1)).sum()), n_attacks))

accuracy           : 0.8030   <- looks fine
recall (malicious) : 0.0000   <- catches nothing
attacks caught     : 0 of 111529


## 2. The Decision Tree — entropy, unpruned

- `criterion='entropy'` → splits chosen by **information gain**, the formulation in AIMA Section 18.3.4. (`gini` yields near-identical trees; entropy is chosen so the textbook math is what actually runs.)
- `max_depth=None` → **unpruned**: grow until every leaf is pure. The honest baseline — and the train-vs-test gap will make overfitting concrete.
- `random_state=42` → reproducibility (`Section 7 #9`), so the SVM compares fairly later.
- **No `class_weight`** — that is the *next* experiment (`Section 7 #12`), not this one.

Fitting an unpruned tree on 2.26M rows takes a few minutes. Expect a deep tree with many leaves.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from time import perf_counter

tree = DecisionTreeClassifier(
    criterion='entropy',   # information gain -- AIMA Section 18.3.4
    max_depth=None,        # unpruned -- the true baseline
    random_state=42,       # foundation.md Section 7 #9
)

t0 = perf_counter()
tree.fit(X_train, y_train)
print('fit in %.1fs' % (perf_counter() - t0))
print('depth: %d | leaves: %d' % (tree.get_depth(), tree.get_n_leaves()))

fit in 38.0s
depth: 67 | leaves: 3109


## 3. Evaluate — accuracy first, then the truth

Read these in order:

1. **Accuracy** — the seductive single number. Compare it against the dummy's.
2. **Train vs test** — the gap an unpruned tree pays for memorising.
3. **Confusion matrix** — where the errors actually live.
4. **Per-class recall** — recall on malicious *is the detection rate*. `FN` = a missed intrusion, `FP` = a false alarm; in an IDS the FN costs more (`Section 11`).

In [4]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = tree.predict(X_test)

print('accuracy  test : %.4f   <- the seductive number' % accuracy_score(y_test, y_pred))
print('accuracy  train: %.4f   <- unpruned trees memorise; mind the gap' % accuracy_score(y_train, tree.predict(X_train)))

cm = confusion_matrix(y_test, y_pred)
print('\nconfusion matrix (rows = actual, cols = predicted):')
print(pd.DataFrame(cm, index=['actual benign', 'actual malicious'],
                       columns=['pred benign', 'pred malicious']))

print('\nFN (missed intrusions): %d   |   FP (false alarms): %d' % (cm[1, 0], cm[0, 1]))
print('\n' + classification_report(y_test, y_pred, target_names=['benign', 'malicious'], digits=4))

accuracy  test : 0.9988   <- the seductive number
accuracy  train: 0.9998   <- unpruned trees memorise; mind the gap

confusion matrix (rows = actual, cols = predicted):
                  pred benign  pred malicious
actual benign          454287             310
actual malicious          378          111151

FN (missed intrusions): 378   |   FP (false alarms): 310

              precision    recall  f1-score   support

      benign     0.9992    0.9993    0.9992    454597
   malicious     0.9972    0.9966    0.9969    111529

    accuracy                         0.9988    566126
   macro avg     0.9982    0.9980    0.9981    566126
weighted avg     0.9988    0.9988    0.9988    566126

